# Fresno State Bin Placement Optimizer
**Part of "What's in the Bin? A Student-Run Plastic Pilot"**
CBIO Collaborative Innovation Design Challenge (BEAM Circular), "Return to Earth"
Eric Pineda Ramirez and Ronnie Pabelonio, Fresno State

**Our project in one line:** single-use plastic in the wrong bin is dragging Fresno State's recycling down, so we propose a one-semester, student-run pilot that tracks which plastic items end up in the wrong bin, sends that tally to Dining Services, and feeds the measurements back into this placement model.

This notebook tells our story with data in four parts:
1. **Diagnosis:** what happened to recycling at Fresno State (real hauling data, 2018 to mid-2022).
2. **Today's layout:** where the receptacles are, from Facilities' 2021 map.
3. **The model:** where receptacles should go to sit closest to where people generate waste.
4. **The pilot:** the data we'd collect next, so the model runs on measured numbers instead of estimates.

**Honesty rules we follow everywhere:**
- The model reports **projected coverage**: the share of estimated waste-generating foot traffic with a receptacle within reach. It is **not** measured recycling or diversion.
- Every estimated number is labeled **PLACEHOLDER** (all of them live in `binopt/config.py`).
- The latest hauling data available to us runs through **June 2022**.

**To run it:** Runtime > Run all. The first cell sets everything up.

## 0. Setup
**Simple:** download our code from GitHub, install the libraries, and copy in the private Facilities files.
**Technical:** in Colab, the cell clones the repo, installs it as a package (`pip install -e .`), mounts Google Drive and copies the private files into `data/private/`. On a laptop it just uses the repo you're in.

The Facilities files are private (the repo is public), so they live in Google Drive, in a folder named **`CoverageBin private data`** with:
`2018-2020 Tonnage Report.xlsx` and `RecepticlesMap-BW-lighter 2021.pdf`.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/Rondo12/CoverageBinCA---CBIO-Collaborative-Innovation-Design-Challenge.git"
BRANCH = "main"
PRIVATE_DRIVE_FOLDER = "/content/drive/MyDrive/CoverageBin private data"

if IN_COLAB:
    !git clone -q -b {BRANCH} {REPO} /content/repo
    %cd /content/repo
    !pip install -q -e . pymupdf
    from google.colab import drive
    drive.mount("/content/drive")
    !cp -r "{PRIVATE_DRIVE_FOLDER}"/. data/private/
else:
    # Running locally from notebooks/: move up to the repo root.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
print("Working in", os.getcwd())
print("Private files:", sorted(os.listdir("data/private")))

## 1. Diagnosis: recycling fell from 46% to 20%
**Data:** Mid Valley Disposal's monthly tonnage report for Fresno State, shared by Facilities (Jan 2018 to Jun 2022).
**Recycling share** = recycling ÷ (trash + recycling): of what went into trash and recycling dumpsters, how much went to recycling. Green waste (mostly tree trimmings) is left out.

In [ ]:
from binopt.tonnage import load_tonnage, yearly_summary
monthly = load_tonnage()
yearly = yearly_summary(monthly)
yearly

In [ ]:
from binopt.figures import tonnage_figure
from IPython.display import Image, display
display(Image(filename=str(tonnage_figure(monthly, yearly)), width=900))

**What this shows**
- In 2018, 46% of trash + recycling tonnage was recycling. By 2021 it was 20%: recycling fell from about **36 to 14 tons a month**.
- Trash rose sharply in 2019, before COVID.
- Over the same period Mid Valley proposed a layout with **more trash and fewer recycling dumpsters** (their "Current" 27 trash / 38 recycling vs. "Proposed" 41 / 30), and the 2021 map shows 40 / 29.
- Mid Valley's 2018 to 2019 site-visit photos show **bagged trash inside recycling dumpsters** at Madden Library, McLane Hall and Science II. Their notes say "purity of diverted waste not known".

**Careful:** this is a pattern, not proof of cause. 2020 and 2021 were COVID years, and we can't confirm exactly when the layout changed. That's exactly the data the pilot would collect.

## 2. Today's layout: the 2021 receptacle map
Facilities' 2021 map is a drawing, not GPS data. `binopt/digitize.py`:
1. **Finds** each marker by its color: yellow dots (receptacles people use), red / blue / green squares (trash / recycling / organics dumpsters).
2. **Pins** the drawing to real GPS with 7 street corners that appear both on the map and in OpenStreetMap (like pinning a paper map onto Apple Maps at a few corners).

Positions are good to about 10 to 20 m (printed below), enough to tell which walkway a receptacle is on.

In [ ]:
from binopt import digitize
points = digitize.main()

## 3. The model
**The question:** given a fixed number of receptacles, where should they go so the most people have one within walking distance?

**Demand (PLACEHOLDER):** each building, food spot and parking lot gets a weight from OpenStreetMap: footprint × floors × a use factor (dining and food high, classrooms medium, offices low). It's an estimate of waste-generating foot traffic, in relative units.

**The algorithm:** the *Maximal Covering Location Problem* (Church & ReVelle, 1974), solved exactly with PuLP:

$$\max \sum_i w_i\,y_i \quad\text{subject to}\quad \sum_j x_j \le p,\qquad y_i \le \sum_{j \in N_i} x_j,\qquad x_j,\,y_i \in \{0,1\}$$

- $x_j = 1$ if a receptacle goes at spot $j$; $y_i = 1$ if demand point $i$ is covered; $w_i$ is its weight.
- $p$ = number of receptacles; $N_i$ = spots within the walking radius of point $i$.
- **Realistic version:** today's receptacles are also spots, and Facilities may **move at most $m$** of them: $\sum_{j \in \text{today}} (1 - x_j) \le m$.
- **Sanity check:** a simple greedy method (keep adding the spot that covers the most uncovered demand) must never beat the exact answer. Tests in `tests/test_model.py` check the model against hand-worked answers.

In [ ]:
from binopt import scenarios
results = scenarios.main()

**Reading the results**
- **Academic core** (campus without student housing, event venues and parking lots): today's 88 receptacles already reach about **90%** of estimated demand within 50 m. Moving just **5** raises it to about **96%**. After ~10 moves there's nothing left to gain.
- The answer holds at **30, 50 and 75 m**, so it isn't an artifact of one walking distance.
- **What this tells us:** in the core, *where* the receptacles are is mostly fine. Placement alone doesn't explain recycling falling from 46% to 20%. The bigger levers are **what goes in the bins** (contamination) and the **dumpster mix**, which is what the pilot measures.
- We don't report "the fewest bins needed": coverage ignores how much a bin holds, so that number would mislead.

In [ ]:
from binopt.plan import make_plan
from binopt.figures import plan_map_figure
plan = make_plan(radius_m=50, max_moves=5)
print(f"Academic core: {plan.today_core:.1%} -> {plan.after_core:.1%} projected coverage")
print("New spots near:", ", ".join(plan.moves_to["near"]))
display(Image(filename=str(plan_map_figure(plan)), width=1000))

Interactive version (switch layers on and off, satellite view top right):

In [ ]:
from binopt.maps import plan_map
plan_map(plan)

### Gaps to ask Facilities about
These places have **no receptacle within 50 m on the 2021 map**. They may be served by someone else (University Housing, event staff), or they may be real gaps. We don't put them in the headline numbers; we ask.

In [ ]:
g = plan.gaps[plan.gaps["group"].notna()]
summary = g.groupby("group").agg(places=("demand_id", "count"), share_of_demand=("share", "sum"))
summary["share_of_demand"] = (summary["share_of_demand"] * 100).round(1).astype(str) + "%"
display(summary)
print("Named places:", ", ".join(n for n in g["name"].dropna() if n))

## 4. The pilot: the data that's missing
The 2020 zero-waste goal was called "vague" and "purity of diverted waste not known." The pilot fills that gap with **one semester of student-collected data, at almost no cost to Facilities**:

1. **Sites (4):** Madden Library, McLane Hall, Science II (where Mid Valley photographed contamination) and the University Dining Hall (food ware).
2. **Weekly log** (Google Form, about 5 minutes per site): fill level of each trash and recycling container, a contamination photo, and the top plastic items seen in the wrong bin.
3. **Two weigh-and-sort audits** (start and end of semester), only with Facilities' approval and protective gear.
4. **One change to test:** pair trash and recycling side by side with clear "no bags in recycling" signs at two sites; keep two as controls.
5. **The tally goes to Dining Services:** if one single-use item (a lid, a cup, a clamshell) lands in the wrong bin week after week, Dining can switch it to a reusable or easier-to-recycle version. Compostable only where organics collection exists.
6. **Back into the model:** measured fill rates and contamination replace the PLACEHOLDER weights, and the model reruns. Real diversion (tons), not just coverage.

**Who runs it:** Eric and Ronnie lead it, with a few student volunteers for the weekly logs.
**One-year plan:** Spring 2027 pilot; Summer 2027 rerun the model and report to Facilities and Dining; Fall 2027 Facilities moves 5 receptacles and expands the signs if contamination dropped.
**Success looks like:** contamination down at the paired sites vs. the controls, one food ware item swapped based on the tally, and a recommended layout Facilities can act on, with no extra pickups.

## Sources
- Mid Valley Disposal tonnage report 2018 to Jun 2022, dumpster maps (Current / Proposed / 2021), and 2018 to 2019 site-visit photos and notes, shared by Fresno State Facilities waste staff.
- Fresno State 2021 receptacle map (Facilities).
- Map data © OpenStreetMap contributors (ODbL).
- Church, R. & ReVelle, C. (1974). The maximal covering location problem. *Papers of the Regional Science Association*, 32, 101 to 118.
- EPA Sustainable Materials Management Web Academy, April 2022 (campus waste characterization methods).